# Build a WLCF Emulator, zs9, 300 Samples

This notebook generates a Latin-hypercube cosmology grid, runs WLCF for each cosmology, trains a neural-network emulator, and saves the weights used by `use_wlcf_emulator.ipynb`.

The implementation lives in `emulator.py`; this notebook is the runnable interface.

## Setup

The script defines the grid, paths, WLCF runner, dataset loader, neural-network training, and emulator serialization.

In [ ]:
%matplotlib inline

from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_tests_dir(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "emulator.py").exists():
            return path
        if (path / "tests" / "emulator.py").exists():
            return path / "tests"
    raise FileNotFoundError("Could not find tests/emulator.py. Open this notebook from the repository or tests folder.")


TEST_DIR = find_tests_dir()
MODULE_PATH = TEST_DIR / "emulator.py"
spec = importlib.util.spec_from_file_location("emulator", MODULE_PATH)
flow = importlib.util.module_from_spec(spec)
sys.modules["emulator"] = flow
spec.loader.exec_module(flow)

config = flow.EmulatorConfig()
paths = flow.make_paths(config)

print(f"Repository: {paths.repo_root}")
print(f"Working directory: {paths.test_dir}")
print(f"Output folder: {paths.emulator_dir}")
print(f"Dataset tag: {config.dataset_tag}")
print(f"Loaded module: {MODULE_PATH}")

## Grid Settings

In [ ]:
print("Grid settings")
print(f"  OMEGAM_RANGE        : {config.omegam_range}")
print(f"  H_RANGE             : {config.h_range}")
print(f"  LOGAS_RANGE         : {config.logas_range}")
print(f"  GRID_MODE           : {config.grid_mode}")
print(f"  N_EVALUATIONS       : {config.n_evaluations}")
print(f"  MOMENTS             : {config.moments}")
print(f"  WLCF THREADS        : {config.number_threads}")
print(f"  OUTPUT              : {paths.emulator_dir}")

## Build the Cosmology Grid

This writes a fresh grid CSV into the standalone output folder.

In [ ]:
grid = flow.build_cosmology_grid(config, paths, save=True)
print(f"Saved grid to {paths.grid_path}")
print(f"Grid shape: {grid.shape}")
grid.describe()[["Omega_m", "h", "logAs"]]

## Generate WLCF Vectors

This is the long-running step. It generates the CAMB linear power spectrum and then runs WLCF for each cosmology. Existing target vectors are skipped, so the notebook can resume after interruption.

In [ ]:
RUN_WLCF_GRID = True

if RUN_WLCF_GRID:
    counts = flow.generate_wlcf_grid(config, paths, grid)
    print("Finished grid generation:", counts)
else:
    print("RUN_WLCF_GRID is False; using existing vectors in", paths.vector_dir)

## Load and Interpolate the Generated Dataset

The generated WLCF vectors are kept in their native 128-point theta grid. When the dataset is loaded, each multipole matrix is interpolated in log-theta to the new 20 theta-bin centers before it enters the neural-network training set.

In [ ]:
data = flow.load_generated_dataset(config, paths, grid)
print(f"Dataset: X={data.X.shape}, y={data.y.shape}")
print(f"Train/validation/test: {len(data.X_train)}, {len(data.X_val)}, {len(data.X_test)}")
print(f"Target shape after interpolation: {data.target_shape}")
print(f"Theta arcmin range: {data.theta_arcmin.min():.3g} to {data.theta_arcmin.max():.3g}")
print(f"Theta bin centers [arcmin]: {np.array2string(data.theta_arcmin, precision=1)}")

fig_mask, fit_mask = flow.plot_new_binning_mask()
print("New binning mask entries per moment:", int(fit_mask.sum()))
plt.show()

## Train and Save the Emulator

The training uses one scikit-learn `MLPRegressor` per multipole. Each network maps `(Omega_m, h, logAs)` to the new-binned `zeta_m` matrix for one value of `m`; the saved emulator concatenates those per-multipole predictions when evaluated.

In [ ]:
TRAIN_EMULATOR = True

if TRAIN_EMULATOR:
    result = flow.train_emulator(config, paths, data)
    print(f"Saved emulator weights to {result.weights_path}")
    display(pd.DataFrame(result.training_report))
else:
    raise RuntimeError("Set TRAIN_EMULATOR=True to create weights for the usage notebook.")

## Test-Set Precision

The first plot summarizes the test-set relative error. The second compares one held-out WLCF target directly against the emulator prediction using the same visual style as `example.ipynb`: shared logarithmic color scale for `|zeta_m|`, log theta axes, and a residual row.

In [ ]:
metrics = flow.summarize_test_error(result)
for key, value in metrics.items():
    print(f"{key:28s}: {value:.6g}")

fig_summary, metrics_df = flow.plot_test_precision_summary(result)
display(metrics_df)
plt.show()

fig_compare = flow.plot_test_prediction_comparison(
    result,
    sample_index=None,
    moments=tuple(config.moments[:3]),
)
plt.show()

## Basic Saved-Emulator Check

In [ ]:
emulator = flow.WLCFEmulator(paths.weights_path)
center = emulator.center_parameters()
vector = emulator.predict_vector(center, moments=emulator.moments[:3])
print("Loaded weights:", emulator.weights_path)
print("Center parameters:", center)
print("Prediction length for first 3 moments:", vector.size)
print("Prediction finite:", np.isfinite(vector).all())

## Files Created

In [ ]:
print("Grid:", paths.grid_path)
print("Vectors:", paths.vector_dir)
print("Weights:", paths.weights_path)
print("Runtime log:", paths.runtime_log_path)
print("Failed samples log:", paths.failed_samples_path)